In [ ]:
# historic bird data from eBird
import time
from datetime import date, timedelta
import pandas as pd
import requests

API_KEY = ""
HEADERS = {"X-eBirdApiToken": API_KEY}
SPECIES_CODE = "mallar3" 
RESULTS = [] 

# 20 european countries
EUROPEAN_COUNTRIES = {
    'Netherlands': 'NL',
    'Germany': 'DE',
    'France': 'FR',
    'United Kingdom': 'GB',
    'Sweden': 'SE',
    'Denmark': 'DK',
    'Poland': 'PL',
    'Spain': 'ES',
    'Italy': 'IT',
    'Switzerland': 'CH',
    'Austria': 'AT',
    'Belgium': 'BE',
    'Norway': 'NO',
    'Finland': 'FI',
    'Ireland': 'IE',
    'Portugal': 'PT',
    'Czech Republic': 'CZ',
    'Hungary': 'HU',
    'Croatia': 'HR',
    'Greece': 'GR'
}


def get_data(country_name, region_code, current_date):
    # Getting bird observations for a region and date from eBird.
   

    # § LLM Help to convert date object to YYYY/MM/DD
    date_str = current_date.strftime("%Y/%m/%d")
    url = f"https://api.ebird.org/v2/data/obs/{region_code}/historic/{date_str}"
    params = {
        "includeProvisional": "true",
        "hotspot": "false"
    }

    print(f"Start request for {region_code} on {date_str}...")
    
    try:
        # § LLM Help to get request with a 10 second timeout
        response = requests.get(url, headers=HEADERS, params=params, timeout=10)
        # Checking, if the request runs withot errors
        if response.status_code == 200:
            data = response.json()

            # Process each observation record in the response list
            for obs in data:
                if obs.get("speciesCode") == SPECIES_CODE:
                    
                    # § LLM Help to extract the actual bird count
                    raw_count = obs.get("howMany", 1)
                    try:
                        count_value = int(raw_count)
                    except (ValueError, TypeError):
                        count_value = 1

                    # Append a dictionary for easy conversion to a Pandas DataFrame
                    RESULTS.append({
                        "Date": date_str,
                        "Country": country_name,
                        "SpeciesCode": SPECIES_CODE,
                        "DuckCount": count_value
                    })
            
            print(f"Added observations for {region_code}")
        else:
            # Giving out the error
            print(f"Error {response.status_code} for {region_code} on {date_str}")
            
    except requests.exceptions.RequestException as e:
        print(f"Request failed: {e}")


# executing our function
if __name__ == "__main__":
    # Defining the time period for the analysis
    years = [2020, 2021, 2022, 2023, 2024]

    # Iterate through each country
    for country_name, country_code in EUROPEAN_COUNTRIES.items():
        print(f"\nAnalysiere Land: {country_name}")
        
        # Iterate through each year
        for year in years:
            start_date = date(year, 3, 1)
           
           # Iterate through all 31 days of March
            for day in range(31):
                # § LLM Help to calculate the exact target date
                target_date = start_date + timedelta(days=day)
                get_data(country_name, country_code, target_date)
                
                # Short pause to not overload the API
                time.sleep(0.2) 

    # Creating our data frame and saving it
    if RESULTS:
        # § LLM Help to convert the list of dictionaries into a structured DataFrame
        df = pd.DataFrame(RESULTS)
        file_name = "europe_ducks_march_2020_2024.csv"
        df.to_csv(file_name, index=False)
        
        print(f"\nData saved as: {file_name}")
        print(df.head())
    else:
        print("\nKeine Daten gefunden.")

    print("------- DONE -------")



Analysiere Land: Netherlands
Start request for NL on 2020/03/01...
1 duck observations added for NL on 2020/03/01
Start request for NL on 2020/03/02...
1 duck observations added for NL on 2020/03/02
Start request for NL on 2020/03/03...
1 duck observations added for NL on 2020/03/03
Start request for NL on 2020/03/04...
1 duck observations added for NL on 2020/03/04
Start request for NL on 2020/03/05...
1 duck observations added for NL on 2020/03/05
Start request for NL on 2020/03/06...
1 duck observations added for NL on 2020/03/06
Start request for NL on 2020/03/07...
1 duck observations added for NL on 2020/03/07
Start request for NL on 2020/03/08...
1 duck observations added for NL on 2020/03/08
Start request for NL on 2020/03/09...
1 duck observations added for NL on 2020/03/09
Start request for NL on 2020/03/10...
1 duck observations added for NL on 2020/03/10
Start request for NL on 2020/03/11...
1 duck observations added for NL on 2020/03/11
Start request for NL on 2020/03/12.

In [ ]:
import time
from datetime import date, timedelta
import pandas as pd
import requests

API_KEY = ''
HEADERS = {'X-eBirdApiToken': API_KEY}
SPECIES_CODE = 'mallar3' 
RESULTS = []

# 20 european countries
EUROPEAN_COUNTRIES = {
    'Netherlands': 'NL',
    'Germany': 'DE',
    'France': 'FR',
    'United Kingdom': 'GB',
    'Sweden': 'SE',
    'Denmark': 'DK',
    'Poland': 'PL',
    'Spain': 'ES',
    'Italy': 'IT',
    'Switzerland': 'CH',
    'Austria': 'AT',
    'Belgium': 'BE',
    'Norway': 'NO',
    'Finland': 'FI',
    'Ireland': 'IE',
    'Portugal': 'PT',
    'Czech Republic': 'CZ',
    'Hungary': 'HU',
    'Croatia': 'HR',
    'Greece': 'GR'
}


def get_recent_data(country_name, country_code):
    # Getting bird observations for a region for the last 30 days.
   
    url = f"https://api.ebird.org/v2/data/obs/{country_code}/recent/{SPECIES_CODE}"
    params = {
        "back": 30,  # last 30 days
        "includeProvisional": "true"
    }

    try:
        # § LLM Help to get the request with a 10 second timeout
        response = requests.get(url, headers=HEADERS, params=params, timeout=10)
        # Checking, if the request runs withot errors
        if response.status_code == 200:
            observations = response.json()

            date_dict = {}

            # § LLM Help to process each observation record in the response list
            for obs in observations:
                obs_date = obs.get('obsDt', '').split(' ')[0]
                if obs_date not in date_dict:
                    date_dict[obs_date] = []
                date_dict[obs_date].append(obs)

            # § LLM Help to aggregate the observations per date
            for obs_date, obs_list in date_dict.items():
                total_ducks = 0
                locations = set()

                for obs in obs_list:
                    count = obs.get("howMany", 1)
                    # making sure, that count is an int
                    try:
                        total_ducks += int(count)
                    except (ValueError, TypeError):
                        total_ducks += 1

                    loc = obs.get("locName")
                    if loc:
                        locations.add(loc)
                # Append a dictionary for easy conversion to DataFrame
                RESULTS.append({
                    "Date": obs_date,
                    "Country": country_name,
                    "DuckCount": total_ducks,
                    "LocationCount": len(locations)
                })
            
            print(f"Erfolgreich: {len(date_dict)} Tage für {country_name} verarbeitet.")

        else:
            print(f"Error {response.status_code} for {country_name}")
            
    except requests.exceptions.RequestException as e:
        print(f"Request failed for {country_name}: {e}")


# executing our code
if __name__ == "__main__":
    # Iterate through all countrys
    for name, code in EUROPEAN_COUNTRIES.items():
        print(f"Analysiere Land: {name}...")
        get_recent_data(name, code)
        
        # Short pause to not overload our API
        time.sleep(0.5) 

    # Creating our data frame and saving it
    if RESULTS:
        df = pd.DataFrame(RESULTS)
        output_file = "europe_ducks_recent_daily.csv"
        df.to_csv(output_file, index=False)

        print(f"\nSaved as {output_file}")
        print(df.head())
    else:
        print("\nKeine Daten zum Speichern gefunden.")

    print("------ DONE ------")



Analysiere Land: Netherlands

Analysiere Land: Germany

Analysiere Land: France

Analysiere Land: United Kingdom

Analysiere Land: Sweden

Analysiere Land: Denmark

Analysiere Land: Poland

Analysiere Land: Spain

Analysiere Land: Italy

Analysiere Land: Switzerland

Analysiere Land: Austria

Analysiere Land: Belgium

Analysiere Land: Norway

Analysiere Land: Finland

Analysiere Land: Ireland

Analysiere Land: Portugal

Analysiere Land: Czech Republic

Analysiere Land: Hungary

Analysiere Land: Croatia

Analysiere Land: Greece

Saved as europe_ducks_recent_daily.csv
         Date      Country  DuckCount  LocationCount
0  2026-03-04  Netherlands         84             12
1  2026-03-03  Netherlands        706             59
2  2026-03-02  Netherlands        304             49
3  2026-03-01  Netherlands        967             98
4  2026-02-28  Netherlands        220             34
------DONE------
